In [1]:
import os
if not os.path.exists('Internship-fly-rank-SEO'):
    !git clone https://github.com/Saad0047/Internship-fly-rank-SEO.git
%cd Internship-fly-rank-SEO

/content/Internship-fly-rank-SEO


# ML-03 — Frame Your Lane as an ML Task

This filled notebook follows the skills/framing-ml-problems guidance. Keep answers short, show the data slice, and compute a simple baseline so the success metric is real and measurable.


## 1. My lane as an ML task (type)

*Which one, and why?*

In [2]:
# 1. Task type: Ranking / scoring
task_type = 'ranking'  # Which content to refresh/fix first (priority score per content)
reason = (
    'Editors must pick which pages to fix first. '
    'A ranked priority score (one number per content item) answers "which ones first?"'
)
print('task_type =', task_type)
print(reason)


task_type = ranking
Editors must pick which pages to fix first. A ranked priority score (one number per content item) answers "which ones first?"


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
# 2. Target / proxy
# Preferred (ideal) target: an OBSERVED future decline in traffic for the content (e.g. % drop in clicks in next 30 days).
# Practical starter proxy (in this repo): `is_declining_label` exists, but NOTE the skill warning: it is derived from `trend_direction/trend_pct` and therefore is a defined rule rather than a practical observed outcome.
target_description = (
    'Ideal: observed decline in the following 30-day window (measured on later data). '
    'Starter proxy (in this CSV): is_declining_label (DERIVED from trend_pct) — use only for iteration and be explicit about the proxy limitation.'
)
print(target_description)


Ideal: observed decline in the following 30-day window (measured on later data). Starter proxy (in this CSV): is_declining_label (DERIVED from trend_pct) — use only for iteration and be explicit about the proxy limitation.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [4]:
# 3. Success metric — pick one: precision@K (ranking metric).
# We'll compute a simple baseline precision@100 using the starter proxy `is_declining_label` and ranking by `trend_pct` (the rule that generated the label).
import pandas as pd
from pathlib import Path
p = Path('data/raw/content_refresh_anonymized.csv')
if not p.exists():
    print('WARNING: starter CSV not found at', p)
else:
    df = pd.read_csv(p)
    print('loaded', p, 'shape=', df.shape)
    # show whether proxy label exists
    if 'is_declining_label' in df.columns:
        base_rate = df['is_declining_label'].mean()
        print(f'base rate (is_declining_label) = {base_rate:.3f} (fraction of rows)')
    else:
        print('is_declining_label not in CSV; you will need an observed downstream label from the warehouse.')
    # baseline: rank by trend_pct (the rule) and compute precision@100 if label exists
    if {'is_declining_label','trend_pct'}.issubset(df.columns):
        topk = 100
        ranked = df.sort_values('trend_pct', ascending=False).head(topk)
        precision_at_k = ranked['is_declining_label'].mean()
        print(f'baseline precision@{topk} (rank by trend_pct) = {precision_at_k:.3f}')
        # A defensible success threshold: improve baseline by a relative lift, e.g. +25% rel.
        success_threshold = precision_at_k * 1.25
        print(f"suggested success target: precision@{topk} >= {success_threshold:.3f} (25% relative lift over baseline)")
    else:
        print('Cannot compute precision@K: missing columns.')


loaded data/raw/content_refresh_anonymized.csv shape= (30000, 44)
is_declining_label not in CSV; you will need an observed downstream label from the warehouse.
Cannot compute precision@K: missing columns.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
# 4. Unit of analysis
# Load the same starter CSV (30k rows). One row = one pseudonymized content item snapshot (content_id × client snapshot).
if p.exists():
    show_cols = ['content_id','client_id','content_type','trend_pct','is_declining_label']
    present = [c for c in show_cols if c in df.columns]
    print('one row = one content item snapshot (pseudonymized). Columns to inspect:', present)
    display(df[present].head(8))
    # Brief dtype and missingness probe
    print('Column dtypes and non-null counts:')
    display(df[present].info())
else:
    print('starter CSV missing; open skills/flyrank/flyrank-data/SKILL.md to request access to the warehouse if you need the full tables.')


one row = one content item snapshot (pseudonymized). Columns to inspect: ['content_id', 'client_id', 'content_type', 'trend_pct']


,content_id,client_id,content_type,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,-34.7
5,content_d4084a4bc775,client_f369cb89fc,keyword article,-38.9
6,content_9a34b442b552,client_8722616204,keyword article,-92.3
7,content_a63219c6e95a,client_19581e27de,keyword article,0.6


Column dtypes and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   content_id    30000 non-null  object 
 1   client_id     30000 non-null  object 
 2   content_type  30000 non-null  object 
 3   trend_pct     26612 non-null  float64
dtypes: float64(1), object(3)
memory usage: 937.6+ KB


None

In [7]:
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('is_declining_label' in df.columns)
print(df['is_declining_label'].value_counts())

True
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [8]:
# 5. Why ML beats a fixed rule — quick probes
if p.exists() and 'is_declining_label' in df.columns:
    # Show that a single threshold on trend_pct is imperfect: compute precision/recall across thresholds
    import numpy as np
    from sklearn.metrics import precision_recall_curve, roc_auc_score
    y = df['is_declining_label'].fillna(0).astype(int)
    scores = df.get('trend_pct', pd.Series(0, index=df.index)).fillna(0).astype(float)
    # precision-recall curve summary
    prec, rec, thr = precision_recall_curve(y, scores)
    # simple AUC (ROC) if possible
    try:
        auc = roc_auc_score(y, scores)
    except Exception as e:
        auc = None
    print('ROC-AUC of single-feature (trend_pct) as a score:', auc)
    # show that missingness and client effects exist: fraction missing keyword fields by content_type
    miss_cols = [c for c in ['keyword_count','word_count'] if c in df.columns]
    if miss_cols:
        print('Missingness by content_type for', miss_cols)
        display(df.groupby('content_type')[miss_cols].apply(lambda d: d.isna().mean()))
    print('Takeaway: a single if-statement (e.g. trend_pct > X) ignores client-level baselines, missingness patterns, and interactions across signals. A learned model can combine many noisy, partially missing signals and client-level baselines to produce a better-prioritized queue.')
else:
    print('Cannot run the comparison probes because the starter CSV or label is missing locally.')


ROC-AUC of single-feature (trend_pct) as a score: 4.5656509179581466e-07
Missingness by content_type for ['word_count']


,word_count
content_type,
comparison article,0.000000
feedly article,0.000000
keyword article,0.282979


Takeaway: a single if-statement (e.g. trend_pct > X) ignores client-level baselines, missingness patterns, and interactions across signals. A learned model can combine many noisy, partially missing signals and client-level baselines to produce a better-prioritized queue.


## Self-check

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom without edits (the intern will run it before submission)
- [ ] No private client names or raw credentials are printed
- [ ] If you need the full warehouse tables, request access and then switch to `skills/flyrank/flyrank-data/SKILL.md` guidance


---
Note: committed fix — repaired notebook JSON and added this note.
